# On importe nos librairies

In [ ]:
%pip install pandas nltk gensim pyLDAvis

import pandas as pd
import gensim
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from gensim.models import CoherenceModel
from nltk.stem import WordNetLemmatizer, SnowballStemmer
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')

# On crée u on importe le dataset

In [9]:
documents = [
{'titre':'Matchlocal', 'texte':'Lejoueur marque un but et son équipe gagne le match.'},
{'titre':'Saisonsportive', 'texte':'L’entraîneurprépare les joueurs pour la nouvelle saison.'},
{'titre':'Hôpital', 'texte':'Lemédecin suit un patient après le traitement.'},
{'titre':'Santépublique', 'texte':'Lespatients parlent des symptômes et du diagnostic.'},
{'titre':'Débatparlementaire', 'texte':'Leministre présente une loi après le vote.'},
{'titre':'Campagne', 'texte':'Leparti organise un débat politique avant les élections.'}]

documents[1]["titre"]

'Saisonsportive'

# on transforme notre liste de dictionnaires en un DF pandas

In [15]:
df = pd.DataFrame(documents)
df["titre"][2]


# n enregistre nos textes dans une liste python
textes = df["texte"].tolist()
print(textes[0])

Lejoueur marque un but et son équipe gagne le match.


# on passe au stemming avec SnowballStemmer (NLTK)

In [18]:
stemmer = SnowballStemmer('french')

stopwords_fr= STOPWORDS.union({
'les','des','une','dans','pour','avec','après','avant',
'plus','tout','tous','être','avoir'
})

def lemmatize_stemming(token):
    lemme= WordNetLemmatizer().lemmatize(token, pos='n')
    return stemmer.stem(lemme)

def preprocess(text):
    tokens = []
    for token in simple_preprocess(text, deacc=True):
        if token not in stopwords_fr and len(token) > 3:
            tokens.append(lemmatize_stemming(token))
    return tokens

# on apllique le nettoyage au corpus (stopwords removal + lemmatization)

In [19]:
processed_docs= [preprocess(doc) for doc in textes]

for titre, tokens in zip(df['titre'], processed_docs):
    print(titre, '->', tokens)

Matchlocal -> ['lejoueur', 'marqu', 'equip', 'gagn', 'match']
Saisonsportive -> ['joueur', 'nouvel', 'saison']
Hôpital -> ['lemedecin', 'suit', 'patient', 'apre', 'trait']
Santépublique -> ['lespatient', 'parlent', 'symptom', 'diagnostic']
Débatparlementaire -> ['leministr', 'present', 'apre', 'vot']
Campagne -> ['lepart', 'organis', 'debat', 'polit', 'elect']


# on construit le dictionnaire Gensim avec identifiant numérique pour chaque mot

In [21]:
dictionary= gensim.corpora.Dictionary(processed_docs)

print(dictionary.token2id)
print('Nombre de mots gardés :', len(dictionary))

{'equip': 0, 'gagn': 1, 'lejoueur': 2, 'marqu': 3, 'match': 4, 'joueur': 5, 'nouvel': 6, 'saison': 7, 'apre': 8, 'lemedecin': 9, 'patient': 10, 'suit': 11, 'trait': 12, 'diagnostic': 13, 'lespatient': 14, 'parlent': 15, 'symptom': 16, 'leministr': 17, 'present': 18, 'vot': 19, 'debat': 20, 'elect': 21, 'lepart': 22, 'organis': 23, 'polit': 24}
Nombre de mots gardés : 25


In [22]:
# Paramètres souples pour ce petit corpus de “demon”
dictionary.filter_extremes(
no_below=1, # garder les mots présents dans au moins 1 document
no_above=0.8, # supprimer les mots présents dans plus de 80% des documents
keep_n=1000 # garder au maximum 1000 mots
)
print('Vocabulaire final :', len(dictionary))

Vocabulaire final : 25


# On convertit en Bag Of Word pour avoir l'id + son nombre d'occurences

In [23]:
bow_corpus= [dictionary.doc2bow(doc) for doc in processed_docs]
for i, bow in enumerate(bow_corpus[:3]):
    print(df.loc[i, 'titre'], '->', bow)

Matchlocal -> [(0, 1), (1, 1), (2, 1), (3, 1), (4, 1)]
Saisonsportive -> [(5, 1), (6, 1), (7, 1)]
Hôpital -> [(8, 1), (9, 1), (10, 1), (11, 1), (12, 1)]


# on construit le modèle LDA

In [24]:
lda_model= gensim.models.LdaModel(
corpus=bow_corpus,
id2word=dictionary,
num_topics=3,
random_state=42,
passes=50,
iterations=100,
alpha='auto',
eta='auto'
)
lda_model.save("modele_lda.gensim")
dictionary.save("dictionnaire_lda.gensim")
# lda_model= gensim.models.LdaModel.load("modele_lda.gensim")
# dictionary= gensim.corpora.Dictionary.load("dictionnaire_lda.gensim")

# on affiche et on renomme les topics 

In [25]:
for idx, topic in lda_model.print_topics(num_words=6):
    print(f'Topic{idx} -> {topic}')

Topic0 -> 0.082*"equip" + 0.082*"gagn" + 0.082*"lejoueur" + 0.082*"marqu" + 0.082*"match" + 0.082*"saison"
Topic1 -> 0.077*"apre" + 0.077*"lemedecin" + 0.077*"patient" + 0.077*"suit" + 0.077*"lespatient" + 0.077*"trait"
Topic2 -> 0.077*"vot" + 0.077*"lepart" + 0.077*"apre" + 0.077*"polit" + 0.077*"elect" + 0.077*"present"


In [26]:
topic_names= {
0: "Sport",
1: "Santé",
2: "Politique"
}

for i, bow in enumerate(bow_corpus):
    distrib= lda_model.get_document_topics(bow)
    print(df.loc[i, 'titre'])
    for topic_id, score in distrib:
        print(" -", topic_names[topic_id], ":", round(score, 3))

Matchlocal
 - Sport : 0.981
Saisonsportive
 - Sport : 0.97
 - Santé : 0.015
 - Politique : 0.015
Hôpital
 - Santé : 0.981
Santépublique
 - Sport : 0.011
 - Santé : 0.977
 - Politique : 0.012
Débatparlementaire
 - Sport : 0.011
 - Santé : 0.012
 - Politique : 0.977
Campagne
 - Politique : 0.981


# on affiche le topic dominant pour chaque document

In [27]:
for i, bow in enumerate(bow_corpus):
    distrib= lda_model.get_document_topics(bow)
    print(df.loc[i, 'titre'], '->', distrib)

Matchlocal -> [(0, np.float32(0.9814112))]
Saisonsportive -> [(0, np.float32(0.96958286)), (1, np.float32(0.015208677)), (2, np.float32(0.015208527))]
Hôpital -> [(1, np.float32(0.9814823))]
Santépublique -> [(0, np.float32(0.011449558)), (1, np.float32(0.97701275)), (2, np.float32(0.011537666))]
Débatparlementaire -> [(0, np.float32(0.011449558)), (1, np.float32(0.011537779)), (2, np.float32(0.97701263))]
Campagne -> [(2, np.float32(0.98148227))]


# on calcule la cohérence

In [28]:
coherence_model_lda= CoherenceModel(
model=lda_model,
texts=processed_docs,
dictionary=dictionary,
coherence='c_v'
)
coherence_lda= coherence_model_lda.get_coherence()
print('CoherenceScore:', coherence_lda)

CoherenceScore: 0.5865532469498976


# nous pouvons également tester plusieurs nombres de topics pour voir comment évolue nos scores de cohérence

In [29]:
scores = []
for k in range(2, 6):
    model = gensim.models.LdaModel(
bow_corpus, id2word=dictionary, num_topics=k,
random_state=42, passes=50, iterations=100
)
    cm = CoherenceModel(model=model, texts=processed_docs,
    dictionary=dictionary, coherence='c_v')
    scores.append((k, cm.get_coherence()))
print(scores)   

[(2, np.float64(0.5429802094645095)), (3, np.float64(0.5287607474483412)), (4, np.float64(0.5466460209500122)), (5, np.float64(0.5473791832471128))]


# visualisation avec pyLDAvis

In [30]:
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

vis = gensimvis.prepare(lda_model, bow_corpus, dictionary)
pyLDAvis.display(vis)
# ou : pyLDAvis.save_html(vis, 'lda_visualisation.html')